# Chapter 1 — What's New in PyTorch 2.x: `torch.compile` demo

Companion notebook for the new 3E Chapter 1 section.
Goal: show a drop-in `torch.compile` on a small MNIST CNN, compare eager vs compiled inference latency on CPU (or CUDA if available).


In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

print("torch:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = SmallCNN().to(device).eval()
compiled = torch.compile(model)


In [ ]:
tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
ds = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)
x = torch.stack([ds[i][0] for i in range(64)]).to(device)

@torch.inference_mode()
def bench(fn, warmup=5, iters=50):
    for _ in range(warmup):
        fn(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1000

eager_ms = bench(model)
compiled_ms = bench(compiled)
print(f"eager:     {eager_ms:.3f} ms / batch")
print(f"compiled:  {compiled_ms:.3f} ms / batch")
print(f"speedup:   {eager_ms / compiled_ms:.2f}x")


## Notes for the book

- On CPU, speedups on tiny models can be small or negative — that is itself a teaching point.
- On GPU (especially with larger transformers / diffusion UNets), `torch.compile` typically lands in the 1.3–2x range.
- Use `torch._dynamo.explain(model)(x)` when debugging graph breaks (revisited in Chapter 12).
